# 04 — Generators

Now we move from **custom iterators** to **generators**.

The key idea is:

> A generator provides a simpler way to create an iterator without manually implementing `__iter__()` and `__next__()`.

In the previous notebook, we built iterator classes by hand. Here, `yield` lets Python handle much of that iterator machinery for us.

## 1. Introduction to Generators

With a custom iterator, we needed several methods and state-management code:

```python
class CountUp:
    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value
```

A generator can express the same idea much more simply:

```python
def count_up(limit):
    for number in range(1, limit + 1):
        yield number
```

The generator handles the iterator machinery for us.

## 2. Why Do We Need Generators?

Generators are useful when we want to produce values **one at a time** rather than creating all values at once.

For example:

```python
def numbers():
    for i in range(5):
        yield i
```

Conceptually:

```text
request value
     ↓
generate value
     ↓
pause
     ↓
request next value
     ↓
generate next value
     ↓
pause
```

Producing values only when they are requested is called **lazy evaluation**.

We will compare iterator and generator behavior more closely later in `07_Iterators_vs_Generators.ipynb`.

## 3. Generator Functions

A function containing `yield` is a **generator function**.

Example:

```python
def numbers():
    yield 1
    yield 2
    yield 3
```

Calling it:

```python
result = numbers()
```

does not immediately give us all three values. Instead, it creates a generator object.

You can inspect its type:

```python
print(type(result))
```

The exact representation of a generator object can vary, but it will identify the object as a generator.

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3

result = numbers()

print(type(result))
print(result)

The important distinction is:

```text
numbers
   ↓
generator function

result
   ↓
generator object
```

The generator object will produce values when we request them.

## 4. The `yield` Keyword

`yield` is the central concept in generators.

Compare:

```python
def normal_function():
    return 10
```

with:

```python
def generator_function():
    yield 10
```

### `return`

```text
function
   ↓
returns value
   ↓
function finishes
```

### `yield`

```text
generator
   ↓
produces value
   ↓
pauses
   ↓
can resume later
```

`yield` produces a value from a generator and pauses the generator's execution until another value is requested.

In [ ]:
def normal_function():
    return 10


def generator_function():
    yield 10


print(normal_function())
print(generator_function())

Notice that the normal function call produces the returned value, while the generator function call produces a generator object. The `yield` value is produced when the generator is advanced.

## 5. Creating a Generator

Let's start with the simplest generator.

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3


gen = numbers()

print(next(gen))
print(next(gen))
print(next(gen))

Expected output:

```text
1
2
3
```

The next request finds that there are no more values and raises `StopIteration`.

In [ ]:
# The generator is exhausted.
# Running this line raises StopIteration.

next(gen)

The generator is consumed as we request values from it. Once it is exhausted, it does not automatically restart.

## 6. Generator Objects

Make the distinction explicit:

```python
def numbers():
    yield 1
    yield 2
    yield 3

gen = numbers()
```

Here:

```text
numbers
   ↓
generator function

gen
   ↓
generator object
```

The generator object is responsible for producing the values.

You can use:

```python
next(gen)
```

and:

```python
for value in gen:
    print(value)
```

A generator is an iterator.

> **Every generator is an iterator.**

In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3


gen = numbers()

print(iter(gen) is gen)

Because a generator is an iterator, it works with both `next()` and `for` loops.

## 7. Using `next()` with Generators

Let's request values step by step.

In [ ]:
def numbers():
    yield 10
    yield 20
    yield 30


gen = numbers()

print(next(gen))
print(next(gen))
print(next(gen))

Expected output:

```text
10
20
30
```

The next call raises `StopIteration` because the generator has no more values.

In [ ]:
# The generator is exhausted.
next(gen)

The basic flow is:

```text
Generator
    ↓
next()
    ↓
yield
    ↓
value
```

## 8. Generators with `for` Loops

A `for` loop is the most common way to consume a generator.

In [ ]:
def numbers():
    yield 10
    yield 20
    yield 30


for number in numbers():
    print(number)

Expected output:

```text
10
20
30
```

Conceptually:

```text
for
 ↓
next()
 ↓
yield 10
 ↓
next()
 ↓
yield 20
 ↓
next()
 ↓
yield 30
 ↓
next()
 ↓
StopIteration
 ↓
loop ends
```

The `for` loop automatically requests values until the generator is exhausted.

## 9. How Generators Remember Their State

This is one of the most important concepts.

Consider a generator with visible progress messages:

In [ ]:
def numbers():
    print("First")
    yield 1

    print("Second")
    yield 2

    print("Third")
    yield 3


gen = numbers()

print(next(gen))

Output:

```text
First
1
```

Now request another value:

In [ ]:
print(next(gen))

Output:

```text
Second
2
```

It does **not** start from the beginning. The generator resumes from where it previously paused.

One more request:

In [ ]:
print(next(gen))

Output:

```text
Third
3
```

The core behavior is:

```text
run
 ↓
yield
 ↓
pause
 ↓
resume
 ↓
yield
 ↓
pause
 ↓
resume
 ↓
...
```

## 10. Generator vs Normal Function

### Normal function

```python
def get_numbers():
    return [1, 2, 3]
```

The function returns the complete list.

### Generator function

```python
def get_numbers():
    yield 1
    yield 2
    yield 3
```

The generator produces values one at a time.

| Normal Function | Generator Function |
|---|---|
| Usually uses `return` | Uses `yield` |
| Returns a result | Produces values |
| Function finishes after `return` | Pauses at `yield` |
| Can return a collection | Produces values lazily |
| Does not maintain generator state | Maintains generator state |

## 11. Generator vs Custom Iterator

Now connect directly to `03_Custom_Iterators`.

### Custom iterator

```python
class CountUp:
    def __init__(self, limit):
        self.current = 1
        self.limit = limit

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration

        value = self.current
        self.current += 1
        return value
```

### Generator

```python
def count_up(limit):
    for number in range(1, limit + 1):
        yield number
```

The generator lets Python handle the iterator machinery for us.

Conceptually:

```text
Custom Iterator
├── __iter__()
├── __next__()
├── state
└── StopIteration

Generator
└── yield
    ├── state
    └── StopIteration handled automatically
```

We are intentionally not going deeply into implementation internals yet.

In [ ]:
def count_up(limit):
    for number in range(1, limit + 1):
        yield number


for number in count_up(5):
    print(number)

The output is:

```text
1
2
3
4
5
```

The generator version expresses the sequence with much less iterator boilerplate.

## 12. Practical Generator Examples

Generators become especially useful when the rule for producing values is easy to express step by step.

### Generate squares

In [ ]:
def squares(n):
    for number in range(1, n + 1):
        yield number ** 2


for value in squares(5):
    print(value)

Expected output:

```text
1
4
9
16
25
```

### Generate even numbers

In [ ]:
def even_numbers(n):
    for number in range(0, n + 1, 2):
        yield number


for number in even_numbers(10):
    print(number)

Expected output:

```text
0
2
4
6
8
10
```

### Generate Fibonacci numbers

A generator is a natural way to produce a Fibonacci sequence progressively.

In [ ]:
def fibonacci(n):
    a = 0
    b = 1

    for _ in range(n):
        yield a
        a, b = b, a + b


for number in fibonacci(8):
    print(number)

Expected output:

```text
0
1
1
2
3
5
8
13
```

The generator computes each next value as the sequence is consumed.

## 13. Infinite Generators

Generators can also represent sequences that do not have a natural endpoint.

In [ ]:
def infinite_numbers():
    number = 1

    while True:
        yield number
        number += 1


numbers = infinite_numbers()

print(next(numbers))
print(next(numbers))
print(next(numbers))
print(next(numbers))

Expected output:

```text
1
2
3
4
```

The generator does not need to store all future values.

### Important warning

Do not consume an infinite generator with an unrestricted loop:

```python
for number in infinite_numbers():
    print(number)
```

There is no natural stopping condition.

Instead, control how many values you request:

In [ ]:
numbers = infinite_numbers()

for _ in range(5):
    print(next(numbers))

Expected output:

```text
1
2
3
4
5
```

## 14. Common Mistakes

### Mistake 1 — Using `return` when you need multiple values

This does not produce multiple values:

```python
def numbers():
    return 1
    return 2
```

Use `yield` when values should be produced one at a time:

```python
def numbers():
    yield 1
    yield 2
```

### Mistake 2 — Forgetting that generators are consumed

```python
gen = (x for x in range(3))

print(list(gen))
print(list(gen))
```

Output:

```text
[0, 1, 2]
[]
```

The generator has been exhausted.

### Mistake 3 — Calling `next()` after exhaustion

```python
def numbers():
    yield 1

gen = numbers()

print(next(gen))
print(next(gen))
```

The second call raises `StopIteration`.

### Mistake 4 — Accidentally creating an infinite loop

```python
def numbers():
    while True:
        yield 1
```

This generator never naturally finishes.

That can be useful, but the code consuming it must provide a stopping condition.

## 15. Summary

Use this mental model:

```text
Generator Function
        │
        │ called
        ▼
Generator Object
        │
        │ next()
        ▼
      yield
        │
        ▼
      value
        │
        ▼
     pauses
        │
        │ next()
        ▼
    resumes
        │
        ▼
      yield
        │
        ▼
      ...
        │
        ▼
 StopIteration
```

### Key points

- A generator function contains `yield`.
- Calling a generator function creates a generator object.
- A generator produces values one at a time.
- `yield` pauses execution.
- The generator resumes from where it paused.
- Generators maintain their state automatically.
- Generators are iterators.
- `for` loops can consume generators.
- Generators can represent infinite sequences.
- Once exhausted, a generator cannot be restarted; create a new generator.

### Curriculum progression

```text
01_Iterables_and_Iterators
        ↓
02_Iterator_Protocol
        ↓
03_Custom_Iterators
        ↓
04_Generators             ← current
        ↓
05_Yield_and_Generator_Functions
        ↓
06_Generator_Expressions
        ↓
07_Iterators_vs_Generators
```

### Scope boundary

`yield` in greater depth belongs in `05_Yield_and_Generator_Functions`.

Generator expressions belong in `06_Generator_Expressions`.

Detailed iterator-vs-generator comparison and efficiency discussion belong in `07_Iterators_vs_Generators`.

The next notebook should therefore focus specifically on **`yield`** and how generator functions pause, resume, and retain state, rather than repeating this basic generator introduction.